# Overview
Output : grapheme_root, vowel_diacritics, consonant_diacritics, and grapheme_id
#### Versions
- SEResNeXt001<br>
Baseline Model Built. <br>
- SEResNeXt002<br>
Cutmix applied. No center crop, Just resize. Minimum Augmentation. SGD to Adam. LR min to 8e-6<br>
32 Split 0 Fold : 0.9843<br>
- SEResNeXt003<br>
Center crop.<br>
32 Split 0 Fold : 0.9817<br>
- SEResNeXt004<br>
No center center crop. Alpha=1.0, Mixup Added. <br>
32 Split 0 Fold : 0.9848<br>

In [1]:
import numpy as np
import pandas as pd
import os, pickle, gc, random, tqdm, cv2, six, math, datetime

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score
from skimage.transform import AffineTransform, warp
import albumentations as A
from collections import OrderedDict

In [2]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data.dataloader import DataLoader
from torch.utils.data.dataset import Dataset

# !pip install ../module/pretrainedmodels-0.7.4/ > /dev/null
import pretrainedmodels

In [3]:
VERSION = 'SEResNeXt004'
TARGET_COLS = ['grapheme_root', 'vowel_diacritic', 'consonant_diacritic', 'grapheme_id']
TRAINING = True
VALIDATION = True
PSEUDO_LABELLING = False
LOCAL_PATH = '../input'
WEIGHT_PATH = '../input/weights'
PREPROCESSED_DATA_PATH = '../input/parquet-to-feather-no-'
IMAGENET_MODEL_WEIGHT_FILE = '../input/weights/se_resnext50_32x4d-a260b3a4.pth'
HEIGHT = 137
WIDTH = 236
IMAGE_SIZE = 128
N_SPLITS = 32
FOLD = [0]
SEED = 9253
BATCH_SIZE = 128
VALIDATION_BATCH_SIZE = 512
EPOCHS = 200
EPOCH_RELEASE = 1
EARLY_STOPPING = 75
BATCH_ACCUMULATION_COUNT = 2
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
NUM_WORKERS = 8

def seed_everything(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
seed_everything(SEED)

# 1. Load Datasets and Perform some EDA

In [4]:
train = pd.read_csv(LOCAL_PATH+'/train.csv')
grapheme_unique_dataset = train[
    ['grapheme_root','vowel_diacritic','consonant_diacritic','grapheme']
].drop_duplicates().reset_index(drop=True)
grapheme_unique = grapheme_unique_dataset['grapheme'].values
train['grapheme_id'] = train['grapheme'].apply(lambda x: np.where(grapheme_unique==x)[0][0])
print('Number of unique graphemes : ', len(grapheme_unique))
print(
    'Number of unique patterns for grapheme_root, vowel_diacritic, and consonant_diacritic : ', 
    train[['grapheme_root','vowel_diacritic','consonant_diacritic'	]].drop_duplicates().shape[0]
)
train.head(2)

Number of unique graphemes :  1295
Number of unique patterns for grapheme_root, vowel_diacritic, and consonant_diacritic :  1292


,image_id,grapheme_root,vowel_diacritic,consonant_diacritic,grapheme,grapheme_id
0,Train_0,15,9,5,ক্ট্রো,0
1,Train_1,159,0,0,হ,1


In [5]:
grapheme_dict = grapheme_unique_dataset.to_dict('index')

with open(LOCAL_PATH+'/grapheme_dict.pickle', 'wb') as f:
    pickle.dump(grapheme_dict, f)

print('Number of graphemes in grapheme_dict : ', len(grapheme_dict))
print('3 Header Samples of grapheme_dict : ')
dict(list(grapheme_dict.items())[0:3])

Number of graphemes in grapheme_dict :  1295
3 Header Samples of grapheme_dict : 


{0: {'grapheme_root': 15,
  'vowel_diacritic': 9,
  'consonant_diacritic': 5,
  'grapheme': 'ক্ট্রো'},
 1: {'grapheme_root': 159,
  'vowel_diacritic': 0,
  'consonant_diacritic': 0,
  'grapheme': 'হ'},
 2: {'grapheme_root': 22,
  'vowel_diacritic': 3,
  'consonant_diacritic': 5,
  'grapheme': 'খ্রী'}}

# 2. Preprocessing : Define Dataset Generator

In [6]:
################
# Crop and Resize
# https://www.kaggle.com/iafoss/image-preprocessing-128x128
################

def bbox(img):
    rows = np.any(img, axis=1)
    cols = np.any(img, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    return rmin, rmax, cmin, cmax

def crop_resize(img0, size=IMAGE_SIZE, pad=16):
    #crop a box around pixels large than the threshold 
    #some images contain line at the sides
    ymin,ymax,xmin,xmax = bbox(img0[5:-5,5:-5] > 80)
    #cropping may cut too much, so we need to add it back
    xmin = xmin - 13 if (xmin > 13) else 0
    ymin = ymin - 10 if (ymin > 10) else 0
    xmax = xmax + 13 if (xmax < WIDTH - 13) else WIDTH
    ymax = ymax + 10 if (ymax < HEIGHT - 10) else HEIGHT
    img = img0[ymin:ymax,xmin:xmax]
    #remove lo intensity pixels as noise
    img[img < 28] = 0
    lx, ly = xmax-xmin,ymax-ymin
    l = max(lx,ly) + pad
    #make sure that the aspect ratio is kept in rescaling
    img = np.pad(img, [((l-ly)//2,), ((l-lx)//2,)], mode='constant')
    return cv2.resize(img,(size,size))

def plain_resize(img, size=IMAGE_SIZE):
    return cv2.resize(img,(size,size))


In [7]:
################
# Augmentations
################

def affine_image(img):
    """
    Args:
        img: (h, w) or (1, h, w)
    Returns:
        img: (h, w)
    """
    if img.ndim == 3:
        img = img[0]
        
    # --- scale ---
    min_scale = 0.8
    max_scale = 1.2
    sx = np.random.uniform(min_scale, max_scale)
    sy = np.random.uniform(min_scale, max_scale)

    # --- rotation ---
    max_rot_angle = 7
    rot_angle = np.random.uniform(-max_rot_angle, max_rot_angle) * np.pi / 180.

    # --- shear ---
    max_shear_angle = 10
    shear_angle = np.random.uniform(-max_shear_angle, max_shear_angle) * np.pi / 180.

    # --- translation ---
    max_translation = 4
    tx = np.random.randint(-max_translation, max_translation)
    ty = np.random.randint(-max_translation, max_translation)

    tform = AffineTransform(scale=(sx, sy), rotation=rot_angle, shear=shear_angle,
                            translation=(tx, ty))
    transformed_image = warp(img, tform)
    assert transformed_image.ndim == 2
    return transformed_image

def add_gaussian_noise(x, sigma):
    x += np.random.randn(*x.shape) * sigma
    x = np.clip(x, 0., 1.)
    return x

def _evaluate_ratio(ratio):
    if ratio <= 0.:
        return False
    return np.random.uniform() < ratio

def apply_aug(aug, image):
    return aug(image=image)['image']

class Transform:
    def __init__(self, affine=True, crop=True, 
                 normalize=True, train=True, 
                 sigma=-1., blur_ratio=0., noise_ratio=0., cutout_ratio=0.,
                 grid_distortion_ratio=0., elastic_distortion_ratio=0., random_brightness_ratio=0.,
                 piece_affine_ratio=0., ssr_ratio=0.):
        self.affine = affine
        self.crop = crop
        self.normalize = normalize
        self.train = train
        self.sigma = sigma / 255.

        self.blur_ratio = blur_ratio
        self.noise_ratio = noise_ratio
        self.cutout_ratio = cutout_ratio
        self.grid_distortion_ratio = grid_distortion_ratio
        self.elastic_distortion_ratio = elastic_distortion_ratio
        self.random_brightness_ratio = random_brightness_ratio
        self.piece_affine_ratio = piece_affine_ratio
        self.ssr_ratio = ssr_ratio

    def __call__(self, example):
        if self.train:
            x, y = example
        else:
            x = example
            
        # --- Augmentation ---
        if self.affine:
            x = affine_image(x)

        # --- Gaussian Noise ---
        if self.sigma > 0.:
            x = add_gaussian_noise(x, sigma=self.sigma)

        # albumentations...
        x = x.astype(np.float32)
        assert x.ndim == 2
        # 1. blur
        if _evaluate_ratio(self.blur_ratio):
            r = np.random.uniform()
            if r < 0.25:
                x = apply_aug(A.Blur(p=1.0), x)
            elif r < 0.5:
                x = apply_aug(A.MedianBlur(blur_limit=5, p=1.0), x)
            elif r < 0.75:
                x = apply_aug(A.GaussianBlur(p=1.0), x)
            else:
                x = apply_aug(A.MotionBlur(p=1.0), x)

        if _evaluate_ratio(self.noise_ratio):
            r = np.random.uniform()
            if r < 0.50:
                x = apply_aug(A.GaussNoise(var_limit=5. / 255., p=1.0), x)
            else:
                x = apply_aug(A.MultiplicativeNoise(p=1.0), x)

        if _evaluate_ratio(self.cutout_ratio):
            # A.Cutout(num_holes=2,  max_h_size=2, max_w_size=2, p=1.0)  # Deprecated...
            x = apply_aug(A.CoarseDropout(max_holes=8, max_height=8, max_width=8, p=1.0), x)

        if _evaluate_ratio(self.grid_distortion_ratio):
            x = apply_aug(A.GridDistortion(p=1.0), x)

        if _evaluate_ratio(self.elastic_distortion_ratio):
            x = apply_aug(A.ElasticTransform(
                sigma=50, alpha=1, alpha_affine=10, p=1.0), x)

        if _evaluate_ratio(self.random_brightness_ratio):
            # A.RandomBrightness(p=1.0)  # Deprecated...
            # A.RandomContrast(p=1.0)    # Deprecated...
            x = apply_aug(A.RandomBrightnessContrast(p=1.0), x)

        if _evaluate_ratio(self.piece_affine_ratio):
            x = apply_aug(A.IAAPiecewiseAffine(p=1.0), x)

        if _evaluate_ratio(self.ssr_ratio):
            x = apply_aug(A.ShiftScaleRotate(
                shift_limit=0.0625,
                scale_limit=0.1,
                rotate_limit=30,
                p=1.0), x)

        if self.normalize:
            x = (x.astype(np.float32) - 0.0692) / 0.2051
        if x.ndim == 2:
            x = x[None, :, :]
        x = x.astype(np.float32)
        if self.train:
            y = y.astype(np.int64)
            return x, y
        else:
            return x
        

In [8]:
################
# Dataset Generator
# https://www.kaggle.com/corochann/bengali-seresnext-training-with-pytorch/data
################

def prepare_images(data_type='train', indices=[0, 1, 2, 3]):
    assert data_type in ['train', 'test']
    image_list = []
    for i in indices:
        if data_type=='test':
            df = pd.read_parquet(LOCAL_PATH+'/test_image_data_{}.parquet'.format(i))
        else:
            df = pd.read_feather(PREPROCESSED_DATA_PATH+'{}/train_image_data_{}.feather'.format(i,i))
        images = 255 - df.iloc[:, 1:].values.reshape(-1, HEIGHT, WIDTH).astype(np.uint8)
        for image in tqdm.tqdm(images):
            image = (image*(255.0/image.max())).astype(np.uint8)
#             image = crop_resize(image)
            image = plain_resize(image)
            image_list.append(image)
        del df
        gc.collect()
    images = np.stack(image_list)
    del image_list
    gc.collect()
    return images

class DatasetMixin(Dataset):
    def __init__(self, transform=None):
        self.transform = transform

    def __getitem__(self, index):
        """Returns an example or a sequence of examples."""
        if torch.is_tensor(index):
            index = index.tolist()
        if isinstance(index, slice):
            current, stop, step = index.indices(len(self))
            return [self.get_example_wrapper(i) for i in
                    six.moves.range(current, stop, step)]
        elif isinstance(index, list) or isinstance(index, np.ndarray):
            return [self.get_example_wrapper(i) for i in index]
        else:
            return self.get_example_wrapper(index)

    def __len__(self):
        """Returns the number of data points."""
        raise NotImplementedError

    def get_example_wrapper(self, i):
        """Wrapper of `get_example`, to apply `transform` if necessary"""
        example = self.get_example(i)
        if self.transform:
            example = self.transform(example)
        return example

    def get_example(self, i):
        """Returns the i-th example.
        Implementations should override it. It should raise :class:`IndexError`
        if the index is invalid.
        Args:
            i (int): The index of the example.
        Returns:
            The i-th example.
        """
        raise NotImplementedError
        
class BengaliAIDataset(DatasetMixin):
    def __init__(self, images, labels=None, transform=None, indices=None):
        super(BengaliAIDataset, self).__init__(transform=transform)
        self.images = images
        self.labels = labels
        if indices is None:
            indices = np.arange(len(images))
        self.indices = indices
        self.train = labels is not None

    def __len__(self):
        """return length of this dataset"""
        return len(self.indices)

    def get_example(self, i):
        """Return i-th data"""
        i = self.indices[i]
        x = self.images[i]
        # Opposite white and black: background will be white and
        # for future Affine transformation
        x = x.astype(np.float32) / 255.
        if self.train:
            y = self.labels[i]
            return x, y
        else:
            return x
        

# 3. Model Definition

In [9]:
"""
https://www.kaggle.com/satian/seresnext101-pytorch-starter
https://github.com/Cadene/pretrained-models.pytorch/blob/021d97897c9aa76ec759deff43d341c4fd45d7ba/pretrainedmodels/models/senet.py#L369
"""

pretrained_settings = {
    'se_resnext50_32x4d': {
        'imagenet': {
            'file_path': IMAGENET_MODEL_WEIGHT_FILE,
            'input_space': 'RGB',
            'input_size': [3, 224, 224],
            'input_range': [0, 1],
            'mean': [0.485, 0.456, 0.406],
            'std': [0.229, 0.224, 0.225],
            'num_classes': 1000
        }
    }
}

class SEModule(nn.Module):

    def __init__(self, channels, reduction):
        super(SEModule, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, channels // reduction, kernel_size=1,
                             padding=0)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(channels // reduction, channels, kernel_size=1,
                             padding=0)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        module_input = x
        x = self.avg_pool(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        return module_input * x


class Bottleneck(nn.Module):
    """
    Base class for bottlenecks that implements `forward()` method.
    """
    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out = self.se_module(out) + residual
        out = self.relu(out)

        return out


class SEBottleneck(Bottleneck):
    """
    Bottleneck for SENet154.
    """
    expansion = 4

    def __init__(self, inplanes, planes, groups, reduction, stride=1,
                 downsample=None):
        super(SEBottleneck, self).__init__()
        self.conv1 = nn.Conv2d(inplanes, planes * 2, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes * 2)
        self.conv2 = nn.Conv2d(planes * 2, planes * 4, kernel_size=3,
                               stride=stride, padding=1, groups=groups,
                               bias=False)
        self.bn2 = nn.BatchNorm2d(planes * 4)
        self.conv3 = nn.Conv2d(planes * 4, planes * 4, kernel_size=1,
                               bias=False)
        self.bn3 = nn.BatchNorm2d(planes * 4)
        self.relu = nn.ReLU(inplace=True)
        self.se_module = SEModule(planes * 4, reduction=reduction)
        self.downsample = downsample
        self.stride = stride


class SEResNetBottleneck(Bottleneck):
    """
    ResNet bottleneck with a Squeeze-and-Excitation module. It follows Caffe
    implementation and uses `stride=stride` in `conv1` and not in `conv2`
    (the latter is used in the torchvision implementation of ResNet).
    """
    expansion = 4

    def __init__(self, inplanes, planes, groups, reduction, stride=1,
                 downsample=None):
        super(SEResNetBottleneck, self).__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=1, bias=False,
                               stride=stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, padding=1,
                               groups=groups, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, planes * 4, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * 4)
        self.relu = nn.ReLU(inplace=True)
        self.se_module = SEModule(planes * 4, reduction=reduction)
        self.downsample = downsample
        self.stride = stride


class SEResNeXtBottleneck(Bottleneck):
    """
    ResNeXt bottleneck type C with a Squeeze-and-Excitation module.
    """
    expansion = 4

    def __init__(self, inplanes, planes, groups, reduction, stride=1,
                 downsample=None, base_width=4):
        super(SEResNeXtBottleneck, self).__init__()
        width = math.floor(planes * (base_width / 64)) * groups
        self.conv1 = nn.Conv2d(inplanes, width, kernel_size=1, bias=False,
                               stride=1)
        self.bn1 = nn.BatchNorm2d(width)
        self.conv2 = nn.Conv2d(width, width, kernel_size=3, stride=stride,
                               padding=1, groups=groups, bias=False)
        self.bn2 = nn.BatchNorm2d(width)
        self.conv3 = nn.Conv2d(width, planes * 4, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * 4)
        self.relu = nn.ReLU(inplace=True)
        self.se_module = SEModule(planes * 4, reduction=reduction)
        self.downsample = downsample
        self.stride = stride


class SENet(nn.Module):

    def __init__(self, block, layers, groups, reduction, dropout_p=0.2,
                 inplanes=128, input_3x3=True, downsample_kernel_size=3,
                 downsample_padding=1, num_classes=1000):
        """
        Parameters
        ----------
        block (nn.Module): Bottleneck class.
            - For SENet154: SEBottleneck
            - For SE-ResNet models: SEResNetBottleneck
            - For SE-ResNeXt models:  SEResNeXtBottleneck
        layers (list of ints): Number of residual blocks for 4 layers of the
            network (layer1...layer4).
        groups (int): Number of groups for the 3x3 convolution in each
            bottleneck block.
            - For SENet154: 64
            - For SE-ResNet models: 1
            - For SE-ResNeXt models:  32
        reduction (int): Reduction ratio for Squeeze-and-Excitation modules.
            - For all models: 16
        dropout_p (float or None): Drop probability for the Dropout layer.
            If `None` the Dropout layer is not used.
            - For SENet154: 0.2
            - For SE-ResNet models: None
            - For SE-ResNeXt models: None
        inplanes (int):  Number of input channels for layer1.
            - For SENet154: 128
            - For SE-ResNet models: 64
            - For SE-ResNeXt models: 64
        input_3x3 (bool): If `True`, use three 3x3 convolutions instead of
            a single 7x7 convolution in layer0.
            - For SENet154: True
            - For SE-ResNet models: False
            - For SE-ResNeXt models: False
        downsample_kernel_size (int): Kernel size for downsampling convolutions
            in layer2, layer3 and layer4.
            - For SENet154: 3
            - For SE-ResNet models: 1
            - For SE-ResNeXt models: 1
        downsample_padding (int): Padding for downsampling convolutions in
            layer2, layer3 and layer4.
            - For SENet154: 1
            - For SE-ResNet models: 0
            - For SE-ResNeXt models: 0
        num_classes (int): Number of outputs in `last_linear` layer.
            - For all models: 1000
        """
        super(SENet, self).__init__()
        self.inplanes = inplanes
        if input_3x3:
            layer0_modules = [
                ('conv1', nn.Conv2d(3, 64, 3, stride=2, padding=1,
                                    bias=False)),
                ('bn1', nn.BatchNorm2d(64)),
                ('relu1', nn.ReLU(inplace=True)),
                ('conv2', nn.Conv2d(64, 64, 3, stride=1, padding=1,
                                    bias=False)),
                ('bn2', nn.BatchNorm2d(64)),
                ('relu2', nn.ReLU(inplace=True)),
                ('conv3', nn.Conv2d(64, inplanes, 3, stride=1, padding=1,
                                    bias=False)),
                ('bn3', nn.BatchNorm2d(inplanes)),
                ('relu3', nn.ReLU(inplace=True)),
            ]
        else:
            layer0_modules = [
                ('conv1', nn.Conv2d(3, inplanes, kernel_size=7, stride=2,
                                    padding=3, bias=False)),
                ('bn1', nn.BatchNorm2d(inplanes)),
                ('relu1', nn.ReLU(inplace=True)),
            ]
        # To preserve compatibility with Caffe weights `ceil_mode=True`
        # is used instead of `padding=1`.
        layer0_modules.append(('pool', nn.MaxPool2d(3, stride=2,
                                                    ceil_mode=True)))
        self.layer0 = nn.Sequential(OrderedDict(layer0_modules))
        self.layer1 = self._make_layer(
            block,
            planes=64,
            blocks=layers[0],
            groups=groups,
            reduction=reduction,
            downsample_kernel_size=1,
            downsample_padding=0
        )
        self.layer2 = self._make_layer(
            block,
            planes=128,
            blocks=layers[1],
            stride=2,
            groups=groups,
            reduction=reduction,
            downsample_kernel_size=downsample_kernel_size,
            downsample_padding=downsample_padding
        )
        self.layer3 = self._make_layer(
            block,
            planes=256,
            blocks=layers[2],
            stride=2,
            groups=groups,
            reduction=reduction,
            downsample_kernel_size=downsample_kernel_size,
            downsample_padding=downsample_padding
        )
        self.layer4 = self._make_layer(
            block,
            planes=512,
            blocks=layers[3],
            stride=2,
            groups=groups,
            reduction=reduction,
            downsample_kernel_size=downsample_kernel_size,
            downsample_padding=downsample_padding
        )
        self.avg_pool = nn.AvgPool2d(7, stride=1)
        self.dropout = nn.Dropout(dropout_p) if dropout_p is not None else None
        self.last_linear = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, planes, blocks, groups, reduction, stride=1,
                    downsample_kernel_size=1, downsample_padding=0):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion,
                          kernel_size=downsample_kernel_size, stride=stride,
                          padding=downsample_padding, bias=False),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, groups, reduction, stride,
                            downsample))
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes, groups, reduction))

        return nn.Sequential(*layers)

    def features(self, x):
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x

    def logits(self, x):
        x = self.avg_pool(x)
        if self.dropout is not None:
            x = self.dropout(x)
        x = x.view(x.size(0), -1)
        x = self.last_linear(x)
        return x

    def forward(self, x):
        x = self.features(x)
        x = self.logits(x)
        return x


def initialize_pretrained_model(model, num_classes, settings):
    assert num_classes == settings['num_classes'], \
        'num_classes should be {}, but is {}'.format(
            settings['num_classes'], num_classes)
    model.load_state_dict(torch.load(settings['file_path']))
    model.input_space = settings['input_space']
    model.input_size = settings['input_size']
    model.input_range = settings['input_range']
    model.mean = settings['mean']
    model.std = settings['std']

def se_resnext50_32x4d(num_classes=1000, pretrained='imagenet'):
    model = SENet(SEResNeXtBottleneck, [3, 4, 6, 3], groups=32, reduction=16,
                  dropout_p=None, inplanes=64, input_3x3=False,
                  downsample_kernel_size=1, downsample_padding=0,
                  num_classes=num_classes)
    if pretrained is not None:
        settings = pretrained_settings['se_resnext50_32x4d'][pretrained]
        initialize_pretrained_model(model, num_classes, settings)
    return model


In [10]:
# model = se_resnext50_32x4d(num_classes=1000, pretrained='imagenet')
# for param in model.parameters():
#     param.requires_grad = False
# model.last_linear = nn.Linear(model.last_linear.in_features, 168+11+7+1295)
# model = model.to(DEVICE)

# 4. Training Utitility Functions

In [11]:
################
# Custom Loss Function
################

def loss_function(y_preds, y_true):
    
    logit1, logit2, logit3, logit4 = y_preds[:,: 168], y_preds[:,168: 168+11], y_preds[:,168+11:168+11+7], y_preds[:,168+11+7:]
    label1, label2, label3, label4 = y_true[:,0], y_true[:,1], y_true[:,2], y_true[:,3]
    
    grapheme_root_loss = nn.CrossEntropyLoss()(logit1, label1) # BCEWithLogitsLoss
    vowel_diacritic_loss = nn.CrossEntropyLoss()(logit2, label2)
    consonant_diacritic_loss = nn.CrossEntropyLoss()(logit3, label3)
    grapheme_id_loss = nn.CrossEntropyLoss()(logit4, label4)
    
    return 0.2 * grapheme_root_loss + 0.1 * vowel_diacritic_loss + \
            0.1 * consonant_diacritic_loss + 0.4 * grapheme_id_loss

In [12]:
################
# 4 logit outputs to predicted 3 labels
################

grapheme_root_matrix = np.zeros((168, 1295))
vowel_diacritic_matrix = np.zeros((11, 1295))
consonant_diacritic_matrix = np.zeros((7, 1295))
for i, value in grapheme_dict.items():
    grapheme_root_matrix[value['grapheme_root']][i] = 1.
    vowel_diacritic_matrix[value['vowel_diacritic']][i] = 1.
    consonant_diacritic_matrix[value['consonant_diacritic']][i] = 1.

def inference_from_raw_logits(y_preds, grapheme_dict=grapheme_dict, grapheme_root_matrix=grapheme_root_matrix, vowel_diacritic_matrix=vowel_diacritic_matrix, consonant_diacritic_matrix=consonant_diacritic_matrix):
    
    logit1, logit2, logit3, logit4 = y_preds[:,: 168], y_preds[:,168: 168+11], y_preds[:,168+11:168+11+7], y_preds[:,168+11+7:]
    threelogits = np.matmul(logit1, grapheme_root_matrix) / 3 + np.matmul(logit2, vowel_diacritic_matrix) / 3 + np.matmul(logit3, consonant_diacritic_matrix) / 3
    grapheme_id_preds = np.argmax(threelogits / 2 + logit4 / 2, axis=1)
    
    f = lambda x: grapheme_dict[x]['grapheme_root']
    f = np.vectorize(f)
    grapheme_root = f(grapheme_id_preds)
    f = lambda x: grapheme_dict[x]['vowel_diacritic']
    f = np.vectorize(f)
    vowel_diacritic = f(grapheme_id_preds)
    f = lambda x: grapheme_dict[x]['consonant_diacritic']
    f = np.vectorize(f)
    consonant_diacritic = f(grapheme_id_preds)
    
    return grapheme_root, vowel_diacritic, consonant_diacritic


In [13]:
################
# Cutmix
################

def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = np.int(W * cut_rat)
    cut_h = np.int(H * cut_rat)

    # uniform
    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

def cutmix(data, labels, alpha=1.0):
    indices = torch.randperm(data.size(0))
    shuffled_data = data[indices]
    labels1 = labels[:,0]
    labels2 = labels[:,1]
    labels3 = labels[:,2]
    labels4 = labels[:,3]
    shuffled_labels1 = labels1[indices]
    shuffled_labels2 = labels2[indices]
    shuffled_labels3 = labels3[indices]
    shuffled_labels4 = labels4[indices]

    lam = np.random.beta(alpha, alpha)
    bbx1, bby1, bbx2, bby2 = rand_bbox(data.size(), lam)
    data[:, :, bbx1:bbx2, bby1:bby2] = data[indices, :, bbx1:bbx2, bby1:bby2]
    # adjust lambda to exactly match pixel ratio
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (data.size()[-1] * data.size()[-2]))

    labels = [labels1, shuffled_labels1, labels2, shuffled_labels2, labels3, shuffled_labels3, labels4, shuffled_labels4, lam]
    return data, labels

def cutmix_loss_function(y_preds, y_true):
    logit1, logit2, logit3, logit4 = y_preds[:,: 168], y_preds[:,168: 168+11], y_preds[:,168+11:168+11+7], y_preds[:,168+11+7:]
    labels1, shuffled_labels1, labels2, shuffled_labels2, labels3, shuffled_labels3, labels4, shuffled_labels4, lam = y_true
    c = nn.CrossEntropyLoss(reduction='mean')
    return 0.2 * (lam * c(logit1, labels1) + (1 - lam) * c(logit1, shuffled_labels1)) + \
            0.1 * (lam * c(logit2, labels2) + (1 - lam) * c(logit2, shuffled_labels2)) + \
            0.1 * (lam * c(logit3, labels3) + (1 - lam) * c(logit3, shuffled_labels3)) + \
            0.4 * (lam * c(logit4, labels4) + (1 - lam) * c(logit4, shuffled_labels4))


In [14]:
################
# Mixup
################

def mixup(data, labels, alpha=1.0):
    indices = torch.randperm(data.size(0))
    shuffled_data = data[indices]
    labels1 = labels[:,0]
    labels2 = labels[:,1]
    labels3 = labels[:,2]
    labels4 = labels[:,3]
    shuffled_labels1 = labels1[indices]
    shuffled_labels2 = labels2[indices]
    shuffled_labels3 = labels3[indices]
    shuffled_labels4 = labels4[indices]

    lam = np.random.beta(alpha, alpha)
    data = data * lam + shuffled_data * (1 - lam)
    
    labels = [labels1, shuffled_labels1, labels2, shuffled_labels2, labels3, shuffled_labels3, labels4, shuffled_labels4, lam]
    return data, labels

def mixup_loss_function(y_preds, y_true):
    logit1, logit2, logit3, logit4 = y_preds[:,: 168], y_preds[:,168: 168+11], y_preds[:,168+11:168+11+7], y_preds[:,168+11+7:]
    labels1, shuffled_labels1, labels2, shuffled_labels2, labels3, shuffled_labels3, labels4, shuffled_labels4, lam = y_true
    c = nn.CrossEntropyLoss(reduction='mean')
    return 0.2 * (lam * c(logit1, labels1) + (1 - lam) * c(logit1, shuffled_labels1)) + \
            0.1 * (lam * c(logit2, labels2) + (1 - lam) * c(logit2, shuffled_labels2)) + \
            0.1 * (lam * c(logit3, labels3) + (1 - lam) * c(logit3, shuffled_labels3)) + \
            0.4 * (lam * c(logit4, labels4) + (1 - lam) * c(logit4, shuffled_labels4))


In [15]:
################
# Training Validating and Predicting Functions
################

def training_with_accumulation(model, train_loader, optimizer, criterion, scheduler, apply_cutmix=False, apply_mixup=False):
    
    model.train()
    avg_loss = 0.
    optimizer.zero_grad()
    
    random_float = np.random.rand()
    if apply_cutmix and (random_float<=0.33):
        apply_cutmix = True
        apply_mixup = False
    elif apply_mixup and (random_float>0.33) and (random_float<0.66):
        apply_cutmix = False
        apply_mixup = True
    
    bar = tqdm.tqdm(
        enumerate(train_loader), 
        total=len(train_loader), 
        postfix={"train_loss":0.0,}
    )
    for idx, batch in bar:
        
        data, labels = batch
        data, labels = data.to(DEVICE), labels.to(DEVICE)
        if apply_cutmix:
            data, labels = cutmix(data, labels)
        elif apply_mixup:
            data, labels = mixup(data, labels)
        
        logits = model(data)
        if apply_cutmix:
            loss = cutmix_loss_function(logits, labels)
        elif apply_mixup:
            loss = mixup_loss_function(logits, labels)
        else:
            loss = criterion(logits, labels)
        loss.backward()
        if (idx + 1) % BATCH_ACCUMULATION_COUNT == 0:    
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        avg_loss += loss.item() / (len(train_loader))
        
        bar.set_postfix(ordered_dict={
            "train_loss":loss.item(),
        })
        del data, labels

    torch.cuda.empty_cache()
    gc.collect()
    
    return avg_loss


def validate_model(model, criterion, val_loader, target_cols=TARGET_COLS, batch_size=VALIDATION_BATCH_SIZE, verbose=False):

    avg_val_loss = 0.
    model.eval()
    if verbose:
        val_loader = tqdm.tqdm(val_loader)
    
    y_preds = np.zeros((len(val_loader.dataset), 168+11+7+1295))
    y_true = np.zeros((len(val_loader.dataset), 4))
    
    with torch.no_grad():
        
        for idx, batch in enumerate(val_loader):
            data, labels = batch
            data, labels = data.to(DEVICE), labels.to(DEVICE)
            
            logits = model(data)
            logits = torch.sigmoid(logits)
            
            avg_val_loss += criterion(logits, labels).item() / len(val_loader)
            logits = logits.detach().cpu().squeeze().numpy()
            labels = labels.detach().cpu().squeeze().numpy()
            y_preds[idx*batch_size : (idx+1)*batch_size] = logits
            y_true[idx*batch_size : (idx+1)*batch_size]  = labels
            
            del logits, labels
            
        torch.cuda.empty_cache()
        gc.collect()
        
        grapheme_root, vowel_diacritic, consonant_diacritic = inference_from_raw_logits(y_preds)
        label1, label2, label3, _ = y_true.T
        
        score = 0.5 * recall_score(grapheme_root, label1, average='macro') + 0.25 * recall_score(vowel_diacritic, label2, average='macro') + 0.25 * recall_score(consonant_diacritic, label3, average='macro')
        if verbose:
            print('grapheme_root score : ', recall_score(grapheme_root, label1, average='macro'))
            print('vowel_diacritic score : ', recall_score(vowel_diacritic, label2, average='macro'))
            print('consonant_diacritic score : ', recall_score(consonant_diacritic, label3, average='macro'))
        
    return avg_val_loss, score


def predict(model, test_loader, target_cols, batch_size=BATCH_SIZE):
    
    test_preds = np.zeros((len(test_loader.dataset), 168+11+7+1295))
    
    model.eval()
    tk0 = tqdm.tqdm(enumerate(test_loader))
    for idx, x_batch in tk0:
        with torch.no_grad():
            data = x_batch
            data = data.to(DEVICE)
            predictions = model(data)
            predictions = torch.sigmoid(predictions)
            test_preds[idx*batch_size : (idx+1)*batch_size] = predictions.detach().cpu().squeeze().numpy()

    grapheme_root, vowel_diacritic, consonant_diacritic = inference_from_raw_logits(test_preds)
    
    return grapheme_root, vowel_diacritic, consonant_diacritic

In [16]:
# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

# criterion = loss_function
# optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=2e-5)

# 5. Training Starts Here

In [17]:
################
# Preprocessing
################

if TRAINING:
    train_images = prepare_images(data_type='train')
    train_labels = train[['grapheme_root','vowel_diacritic','consonant_diacritic','grapheme_id']].values
    

100%|██████████| 50210/50210 [00:15<00:00, 3152.17it/s]


In [ ]:
################
# Training Part
################

if TRAINING and VALIDATION:
    
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fold, (trn_idx, val_idx) in enumerate(skf.split(X=train.image_id, y=train.grapheme_id)):
        
        if fold not in FOLD:
            continue
            
        train_transform = Transform()
        valid_transform = Transform(affine=False)

        train_dataset = BengaliAIDataset(train_images, train_labels, transform=train_transform, indices=trn_idx)
        valid_dataset = BengaliAIDataset(train_images, train_labels, transform=valid_transform, indices=val_idx)
        
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=True)
        valid_loader = DataLoader(valid_dataset, batch_size=VALIDATION_BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False)
        
        model = se_resnext50_32x4d(num_classes=1000, pretrained='imagenet')
        if EPOCH_RELEASE > 0:
            for param in model.parameters():
                param.requires_grad = False
        model.layer0.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        model.avg_pool = nn.AdaptiveAvgPool2d(1)
        model.last_linear = nn.Linear(model.last_linear.in_features, 168+11+7+1295)
        model = model.to(DEVICE)
        model.zero_grad()
        model.to(DEVICE)
        torch.cuda.empty_cache()
        
        model.train()
        
        criterion = loss_function
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=8e-6)
        
        train_start_time = datetime.datetime.now()
        best_score = 0.0
        for epoch in range(EPOCHS):
            epoch_start_time = datetime.datetime.now()
            torch.cuda.empty_cache()

            if epoch == EPOCH_RELEASE:
                for param in model.parameters():
                    param.requires_grad = True
                    
            apply_cutmix = True if np.random.rand() < 0.25 else False

            avg_loss = training_with_accumulation(
                model, train_loader, optimizer, criterion, scheduler, apply_cutmix=apply_cutmix)
            avg_val_loss, val_score = validate_model(model, criterion, valid_loader)

            print("Epoch {} : {} seconds : train loss {:.4f} : valid loss {:.4f} : valid score {:.4f}".format(
                epoch, (datetime.datetime.now() - epoch_start_time).seconds, avg_loss, avg_val_loss, val_score))

            if val_score > best_score:
                best_score = val_score
                torch.save(model.state_dict(), os.path.join("./model_{}_{}.ckpt".format(VERSION, fold)))
                early_stopping_count = 0
            else:
                early_stopping_count += 1
                if early_stopping_count == EARLY_STOPPING:
                    print("Early Stopping : ", epoch)
                    break

        print('-'*20)
        print("Fold {} : Total Training Time {}, Best Score : {}".format(
            fold, datetime.datetime.now()-train_start_time, best_score))
        print('-'*20)
        
        model.load_state_dict(torch.load(os.path.join("./model_{}_{}.ckpt".format(VERSION, fold))))
        avg_val_loss, val_spearmanr = validate_model(
            model, valid_loader, verbose=True)
        best_scores.append(val_score)
        
        del model
        gc.collect()

100%|██████████| 1521/1521 [04:00<00:00,  6.33it/s, train_loss=4.18]


Epoch 0 : 244 seconds : train loss 4.0610 : valid loss 4.2010 : valid score 0.2922


/opt/anaconda3/lib/python3.7/site-packages/sklearn/metrics/_classification.py:1268: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
100%|██████████| 1521/1521 [05:48<00:00,  4.36it/s, train_loss=0.49] 


Epoch 1 : 353 seconds : train loss 0.6484 : valid loss 3.6511 : valid score 0.9408


100%|██████████| 1521/1521 [05:48<00:00,  4.36it/s, train_loss=0.65]  


...

100%|██████████| 1521/1521 [05:48<00:00,  4.36it/s, train_loss=0.144] 


Epoch 3 : 353 seconds : train loss 0.1884 : valid loss 3.6363 : valid score 0.9436


100%|██████████| 1521/1521 [05:49<00:00,  4.36it/s, train_loss=0.649] 


Epoch 4 : 353 seconds : train loss 0.1586 : valid loss 3.6308 : valid score 0.9494


 71%|███████   | 1075/1521 [04:06<01:41,  4.39it/s, train_loss=0.178] 

In [19]:
# just to check
model.load_state_dict(torch.load(os.path.join("./model_{}_{}.ckpt".format(VERSION, fold))))
avg_val_loss, val_score = validate_model(model, criterion, valid_loader)
val_score

0.9849286465868143

In [ ]:
if TRAINING and not VALIDATION:
    
    train_transform = Transform()
    train_dataset = BengaliAIDataset(train_images, train_labels, transform=train_transform)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    model = se_resnext50_32x4d(num_classes=1000, pretrained='imagenet')
    if EPOCH_RELEASE > 0:
        for param in model.parameters():
            param.requires_grad = False
    model.layer0.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    model.avg_pool = nn.AdaptiveAvgPool2d(1)
    model.last_linear = nn.Linear(model.last_linear.in_features, 168+11+7+1295)
    model = model.to(DEVICE)
    model.zero_grad()
    model.to(DEVICE)
    torch.cuda.empty_cache()

    model.train()

    criterion = loss_function
    optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=2e-5)

    train_start_time = datetime.datetime.now()
    best_score = 0.0
    for epoch in range(EPOCHS):
        epoch_start_time = datetime.datetime.now()
        torch.cuda.empty_cache()

        if epoch == EPOCH_RELEASE:
            for param in model.parameters():
                param.requires_grad = True

        avg_loss = training_with_accumulation(
            model, train_loader, optimizer, criterion, scheduler)
        
    torch.save(model.state_dict(), os.path.join("./model_{}.ckpt".format(VERSION)))
    

# 6. Prediction

# 7. Pseudo Labelling

### 7.1 Pseudo Labelling and Training

### 7.2 Prediction After Pseudo Labelling

# 8. Submission